# Task C — amortization at the full 77-column constraint set, self-contained on Colab

Everything runs here, in one go: P_θ is **trained on this device** with the frozen recipe (`frozen()`: 450 epochs, cosine lr 1e-3 → 1e-5, EMA 0.999, batch 512), then the dual, h_ψ, and both arms of the matched-compute comparison. Nothing is loaded from a laptop.

**This P_θ is trained separately from the §5.4 checkpoint, with the same recipe.** That is fine for this experiment: it compares two representations of the *same* projected measure — weighted importance sampling versus the amortized sampler — on one model, and the comparison is internal to whichever model is used. Numbers that must match §5.4 exactly (the headline sweep, the overlay baseline) come from the checkpoint, not from here.

The two arms:
- **arm W (weighted)** — P_θ draws + importance weights from β\*, plus multinomial resampling to produce unweighted paths;
- **arm A (amortized)** — the corrected sampler with the ε-hook `−√(1−ᾱ)·∇log h_ψ`.

Compute is booked for each arm (network evaluations and wall-clock, h_ψ's training included) so "when is it worth converting weights into a sampler" is answered with numbers.

Constraints: 53 calibrated (C3) + 24 held out = 77 columns. The dual is solved on the 53; the 24 are scored, never calibrated. Drive is written after every stage and `SKIP_DONE` resumes. A100 runtime ≈ 45–55 min (training ≈ 12 min of it).

### Cell 0 — clone, pin, and fail fast if anything is stale

In [ ]:
PINNED_COMMIT   = "7dedf6213e3854ca02b491abec2c835635d5a63c"          # the code this notebook was written and dry-run against
NOTEBOOK_VERSION = "amort77-2026.09.23a"
EXPECT_TASKC     = "taskc-2026.09.23a"     # taskc.__version__ this notebook expects

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned commit :", PINNED_COMMIT)
print("checked out   :", HEAD)
print("notebook      :", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), (
    f"checkout did not land on the pinned commit (HEAD={HEAD}, pinned={PINNED_COMMIT}) -- stop")
!git log --oneline -1

### Cell 1 — imports, full fp32, version assertions

TF32 is **off** and matmul precision is `highest`: the reverse chain runs 998 steps and a per-step relative error of ~1e-3 (TF32's mantissa) compounds, which is a candidate explanation for a checkpoint behaving differently on an A100 than on CPU/MPS.

In [ ]:
import os, sys, json, math, pickle, time
import torch

# full fp32 -- set before any model code touches a tensor
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import asdict, replace
import taskc
from taskc.config import CFG, FROZEN, frozen
from taskc.data import build_training_set, make_loader, reference_paths
from taskc.ptheta import (make_schedule, build_model, train_ptheta, save_checkpoint, load_checkpoint,
                          sample_ptheta, save_draw, load_draw, draw_path)
from taskc.gate import run_gate, print_report, summary_dict
from config import q_params, CONSTRAINT_LEVELS, EXOTICS      # taskb

assert taskc.__version__ == EXPECT_TASKC, (
    f"stale notebook or stale checkout: taskc.__version__={taskc.__version__}, expected {EXPECT_TASKC}")
nb_path = "notebooks/taskc_06_amortization_colab.ipynb"
assert NOTEBOOK_VERSION in open(nb_path).read(), (
    "the notebook running here is not the one committed at the pinned commit -- "
    "re-open it from the pinned URL (Colab may be serving a cached copy)")

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("tf32 matmul:", torch.backends.cuda.matmul.allow_tf32, "| tf32 cudnn:", torch.backends.cudnn.allow_tf32,
      "| matmul precision:", torch.get_float32_matmul_precision())
print("taskc:", taskc.__version__, "| torch:", torch.__version__)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_option_pricing"
else:
    DRIVE = os.environ.get("DRY_OUT", "artifacts_colab_local")
os.makedirs(DRIVE, exist_ok=True)

def close(a, b, rtol=1e-6, atol=1e-12):
    """Relative-tolerance float comparison; never ==."""
    return math.isclose(float(a), float(b), rel_tol=rtol, abs_tol=atol)

### Cell 2 — config, resumability, per-stage saving

In [ ]:
RUN_TAG   = "amort77"
LEVEL     = "C3"
SKIP_DONE = True            # resume: reuse any stage already written to Drive
QUICK     = False           # True -> reduced sizes for a plumbing check; NOT for the paper

OUT = os.path.join(DRIVE, "artifacts_" + RUN_TAG); os.makedirs(OUT, exist_ok=True)
RUN = frozen(artifact_dir=OUT)                       # 450 epochs, cosine lr decay, EMA 0.999
if QUICK:
    RUN = replace(FROZEN, artifact_dir=OUT, n_train=20_000, n_ref=20_000, epochs=60, sample_chunk=5_000,
                  draws={k: type(CFG.draws[k])(seed=CFG.draws[k].seed, n=5_000) for k in ("A", "B", "C")},
                  solve_draw=type(CFG.solve_draw)(seed=CFG.solve_draw.seed, n=20_000))
    print("*** QUICK: reduced sizes, results are NOT valid for the paper "
          "(the gate thresholds alone assume 1e5-path draws) ***")

RES_PATH = os.path.join(OUT, "amortization.json")
res = json.load(open(RES_PATH)) if (SKIP_DONE and os.path.exists(RES_PATH)) else dict(
    tag=RUN_TAG, notebook=NOTEBOOK_VERSION, pinned_commit=PINNED_COMMIT, taskc=taskc.__version__,
    device=DEVICE, device_name=(torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE),
    fp32=dict(tf32_matmul=torch.backends.cuda.matmul.allow_tf32, tf32_cudnn=torch.backends.cudnn.allow_tf32,
              matmul_precision=torch.get_float32_matmul_precision()),
    config=asdict(RUN), quick=QUICK, level=LEVEL, timings={}, stages={})
def save():
    json.dump(res, open(RES_PATH, "w"), indent=1, default=float)
save()
print(f"recipe: {RUN.epochs} epochs, lr {RUN.lr} -> {RUN.lr_min} cosine, EMA {RUN.ema_decay}, batch {RUN.batch_size}")
print(f"draws: A {RUN.draws['A'].n:,} | B {RUN.draws['B'].n:,} | C {RUN.draws['C'].n:,} | solve {RUN.solve_draw.n:,}")
print("stages already done:", list(res["stages"]), "->", OUT)

### Stage 1 — train P_θ here, with the EMA assertion and the pre-registered gate retry

`train_ptheta` dispatches to the decay+EMA loop and returns the **EMA** weights, keeping the raw last iterate on `model.raw_state_dict`. The assertion below checks the returned weights really are the EMA ones (they must differ from the raw iterate), because sampling the raw iterate is exactly the defect that produced the level-bias signature.

Gate rule (`DECISIONS.md` §14): on failure, retrain **once** with a different init seed; stop only if the second attempt fails too, printing the full table.

In [ ]:
from taskc.ptheta import train_ptheta_decay_ema

def train_and_gate(cfg, init_seed, attempt):
    ts_ = build_training_set(cfg); sched_ = make_schedule(cfg, device=DEVICE)
    c = replace(cfg, init_seed=init_seed)
    t0 = time.time()
    model_ = build_model(c).to(DEVICE)
    before = {k: v.detach().clone() for k, v in model_.state_dict().items()}
    model_ = train_ptheta(model_, make_loader(ts_.z, c.batch_size, seed=init_seed), sched_, c, device=DEVICE)
    secs = time.time() - t0
    # the returned weights must be the EMA ones, not the raw last iterate
    assert hasattr(model_, "raw_state_dict"), "decay+EMA loop did not run (check cfg.lr_decay / cfg.ema)"
    k0 = next(k for k, v in model_.state_dict().items() if v.dtype.is_floating_point)
    ema_w, raw_w = model_.state_dict()[k0].detach().cpu(), model_.raw_state_dict[k0]
    assert not torch.allclose(ema_w, raw_w), "sampled weights equal the raw last iterate -- EMA was not applied"
    assert not torch.allclose(ema_w, before[k0].cpu()), "weights did not move during training"
    drift = float((ema_w - raw_w).abs().max())
    print(f"  EMA applied: max|EMA - raw last iterate| = {drift:.3e} on '{k0}'")
    A_ = sample_ptheta(model_, sched_, c.draws["A"].n, c.draws["A"].seed, c, DEVICE, verbose=False)
    g = run_gate(A_.z, ts_.std, c, ref=reference_paths(c))
    return model_, ts_, sched_, A_, g, secs, drift

if SKIP_DONE and "ptheta" in res["stages"] and os.path.exists(os.path.join(OUT, RUN.ckpt_name)):
    ts = build_training_set(RUN); sched = make_schedule(RUN, device=DEVICE)
    model, std_ck, _, ck_extra = load_checkpoint(os.path.join(OUT, RUN.ckpt_name), device=DEVICE)
    for f in ("m", "s", "S0", "r", "dt"):
        assert close(getattr(std_ck, f), getattr(ts.std, f)), f"standardizer mismatch on {f}"
    A = load_draw(RUN, "A"); print("resumed P_theta from Drive:", ck_extra)
else:
    attempt, gate, RETRY_SEED = 1, None, RUN.init_seed + 101
    for attempt, seed in ((1, RUN.init_seed), (2, RETRY_SEED)):
        print(f"training attempt {attempt} (init seed {seed})")
        model, ts, sched, A, gate, secs, drift = train_and_gate(RUN, seed, attempt)
        print(f"  attempt {attempt}: gate {'PASSED' if gate.passed else 'FAILED'} ({secs/60:.1f} min)")
        res["stages"].setdefault("attempts", []).append(dict(attempt=attempt, init_seed=seed, passed=bool(gate.passed),
                                                             train_s=secs, ema_drift=drift, gate=summary_dict(gate)))
        res["timings"][f"train_attempt{attempt}_s"] = secs; save()
        if gate.passed:
            break
        print("  gate failed -- retrying once with a different init seed (DECISIONS.md section 14)")
    print_report(gate, columns=not gate.passed)
    assert gate.passed, ("P_theta failed its gate twice on this device; the full table is printed above. "
                         "Stop and report -- do not proceed to the dual.")
    save_checkpoint(os.path.join(OUT, RUN.ckpt_name), model, ts.std, RUN,
                    extra=dict(tag=RUN_TAG, attempt=attempt, device=DEVICE, notebook=NOTEBOOK_VERSION))
    save_draw(RUN, "A", A)
    res["stages"]["ptheta"] = dict(attempts=attempt, gate=summary_dict(gate), ema_drift=drift)
    res["stages"]["gate"] = summary_dict(gate); save()

if "draws" not in res["stages"] or not SKIP_DONE:
    t0 = time.time()
    B = sample_ptheta(model, sched, RUN.draws["B"].n, RUN.draws["B"].seed, RUN, DEVICE, verbose=False); save_draw(RUN, "B", B)
    C = sample_ptheta(model, sched, RUN.draws["C"].n, RUN.draws["C"].seed, RUN, DEVICE, verbose=False); save_draw(RUN, "C", C)
    res["timings"]["draws_BC_s"] = time.time() - t0
    res["stages"]["draws"] = {k: dict(n=int(v.z.shape[0]), seed=v.seed, rejected=v.n_rejected, seconds=v.seconds, device=v.device)
                              for k, v in (("A", A), ("B", B), ("C", C))}
    save()
else:
    B, C = load_draw(RUN, "B"), load_draw(RUN, "C"); print("resumed draws B, C from Drive")
print("draws:", {k: v["n"] for k, v in res["stages"]["draws"].items()}, "| rejections:", {k: v["rejected"] for k, v in res["stages"]["draws"].items()})

### Stage 2 — dual on the 10⁶ solve draw

In [ ]:
from taskc.dual import solve_level, evaluate_on, baseline_on
q = q_params()
if SKIP_DONE and os.path.exists(os.path.join(OUT, f"tilt_{LEVEL}.pkl")) and "dual" in res["stages"]:
    tilt = pickle.load(open(os.path.join(OUT, f"tilt_{LEVEL}.pkl"), "rb")); print("resumed tilt from Drive")
else:
    t0 = time.time()
    S6 = sample_ptheta(model, sched, RUN.solve_draw.n, RUN.solve_draw.seed, RUN, DEVICE, verbose=False); save_draw(RUN, "S6", S6)
    res["timings"]["solve_draw_s"] = time.time() - t0
    t0 = time.time(); tilt, r, cs = solve_level(LEVEL, ts.std.to_paths(S6.z, "S6"), q); res["timings"]["dual_s"] = time.time() - t0
    assert r.converged, "dual did not converge"
    pickle.dump(tilt, open(os.path.join(OUT, f"tilt_{LEVEL}.pkl"), "wb"))
    res["stages"]["dual"] = dict(m=int(cs.m), beta_raw_norm=float(np.linalg.norm(r.beta_raw)),
                                 screen_ok=bool(r.screen["all_ok"]), screen_margin=float(r.screen["margin"].min()),
                                 screen_bite=cs.names[int(np.argmin(r.screen["margin"]))],
                                 ess_solve=float(r.ess_frac), kl=float(r.kl), n_iter=int(r.n_iter)); save()
    print(f"dual {LEVEL}: m={cs.m} |beta_raw|={np.linalg.norm(r.beta_raw):.2f} screen margin {r.screen['margin'].min():.3f} "
          f"({cs.names[int(np.argmin(r.screen['margin']))]}) ESS_solve {r.ess_frac*100:.2f}% ({res['timings']['dual_s']:.0f}s)")
pC = ts.std.to_paths(C.z, "C"); eC = evaluate_on(tilt, pC, q); w = eC["w"]
assert close(eC["E_L"], 1.0, rtol=5e-3), f"E_C[L*] = {eC['E_L']:.5f} is not ~1"
print(f"on draw C: ESS {eC['ess_frac']*100:.2f}%  E_C[L*] {eC['E_L']:.4f}  max weight {eC['max_weight_ratio']:.1f}x uniform")
res["stages"]["dual_on_C"] = dict(ess_C=float(eC["ess_frac"]), E_C_L=float(eC["E_L"]), maxw=float(eC["max_weight_ratio"])); save()

### Stage 3 — h_ψ on draw B

In [ ]:
from taskc.hnet import HNet, train_hnet, tower_curve, h0_vs_L, grad_ratio_curve, save_hnet, load_hnet
from taskc.run_hpsi import targets_for
HPSI = os.path.join(OUT, f"hpsi_{LEVEL}.pt")
LB, LC = targets_for(tilt, ts.std, B.z, q), targets_for(tilt, ts.std, C.z, q)
if SKIP_DONE and os.path.exists(HPSI) and "hpsi" in res["stages"]:
    hnet = load_hnet(HPSI, device=DEVICE); print("resumed h_psi from Drive")
else:
    hnet = HNet(RUN.data_dim, 256, 32, 0.05)
    t0 = time.time(); hlog = train_hnet(hnet, B.z, LB, sched, device=DEVICE, epochs=5 if QUICK else 100, verbose=False)
    res["timings"]["hpsi_s"] = time.time() - t0
    save_hnet(HPSI, hnet, dict(level=LEVEL, device=DEVICE, E_L_B=float(LB.mean()), notebook=NOTEBOOK_VERSION))
    tl = [0, 5, 20, 50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 950, 998]
    tower = tower_curve(hnet, C.z, sched, tl, device=DEVICE); h0 = h0_vs_L(hnet, C.z, LC, sched, device=DEVICE)
    res["stages"]["hpsi"] = dict(E_L_B=float(LB.mean()), L_min=float(LB.min()), L_max=float(LB.max()),
                                 tower={str(t): tower[t] for t in tl}, h0=h0, mse=hlog.epoch_loss,
                                 floor_max=max(hlog.floor_frac), seconds=res["timings"]["hpsi_s"]); save()
    print(f"h_psi {res['timings']['hpsi_s']/60:.1f} min | L* on B: E {LB.mean():.4f} [{LB.min():.3f},{LB.max():.3f}] | "
          f"tower max |E[h]-1| {max(abs(tower[t]['mean']-1) for t in tl):.4f} | h0 vs L*: corr {h0['corr']:.4f} R2 {h0['r2']:.4f} | "
          f"floor {max(hlog.floor_frac):.1e}")

### Stage 4 — the two arms, matched compute

In [ ]:
from taskc.smt import sample_smt, sliced_wasserstein, resample_weighted, compute_budget
from constraints import build, build_heldout
from config import HELDOUT_TESTFUNS, HELDOUT_VANILLAS
from projection import wmean, wmean_se
import evaluation as ev
N_EVAL = RUN.draws["C"].n; spec = CONSTRAINT_LEVELS[LEVEL]
t0 = time.time(); SMT = sample_smt(model, hnet, sched, N_EVAL, 2001, RUN, DEVICE, verbose=False)
res["timings"]["smt_draw_s"] = time.time() - t0
save_draw(replace(RUN, artifact_dir=OUT), f"smt_{LEVEL}", SMT)
pS = ts.std.to_paths(SMT.z, "SMT")
t0 = time.time(); zW = resample_weighted(C.z, w, N_EVAL, seed=2002); res["timings"]["resample_s"] = time.time() - t0
pW = ts.std.to_paths(zW, "weighted-resampled")
per_draw = res["timings"].get("draws_BC_s", 0.0) / 2
budget = dict(armW=compute_budget(RUN.draws["C"].n + RUN.solve_draw.n, sched.t_start + 1,
                                  per_draw + res["timings"].get("solve_draw_s", 0) + res["timings"].get("dual_s", 0) + res["timings"]["resample_s"]),
              armA=compute_budget(N_EVAL, sched.t_start + 1,
                                  res["timings"]["smt_draw_s"] + res["timings"].get("hpsi_s", 0) + per_draw))
res["stages"]["budget"] = budget
print("arm W:", budget["armW"], "\narm A:", budget["armA"])
csS, csC = build(pS, spec["testfuns"], spec["vanillas"], q), build(pC, spec["testfuns"], spec["vanillas"], q)
hoS, hoC = build_heldout(pS, HELDOUT_TESTFUNS, HELDOUT_VANILLAS, q), build_heldout(pC, HELDOUT_TESTFUNS, HELDOUT_VANILLAS, q)
mS, seS = csS.G.mean(0), csS.G.std(0, ddof=1)/np.sqrt(csS.n)
mW = np.array([wmean(csC.G[:, j], w) for j in range(csC.m)]); seW = np.array([wmean_se(csC.G[:, j], w) for j in range(csC.m)])
devS, devW, dSW = (mS-csS.c)/seS, (mW-csC.c)/seW, (mS-mW)/np.hypot(seS, seW)
mhS = hoS.G.mean(0); mhW = np.array([wmean(hoC.G[:, j], w) for j in range(hoC.m)])
dh = (mhS-mhW)/np.hypot(hoS.G.std(0, ddof=1)/np.sqrt(hoS.n), np.array([wmean_se(hoC.G[:, j], w) for j in range(hoC.m)]))
print(f"\ncalibrated ({csS.m}): SMT vs c max {np.abs(devS).max():.2f} SE, within 2 SE {int((np.abs(devS)<=2).sum())}/{csS.m} | "
      f"weighted vs c max {np.abs(devW).max():.2f} SE | SMT vs weighted max {np.abs(dSW).max():.2f} SE, within 2 SE {int((np.abs(dSW)<=2).sum())}/{csS.m}")
print(f"held-out ({hoS.m}): SMT vs weighted max {np.abs(dh).max():.2f} SE, within 2 SE {int((np.abs(dh)<=2).sum())}/{hoS.m}")
exS, exW, exR = ev.price_exotics(pS), eC["exotics"], ev.price_exotics(pW)
for k in EXOTICS:
    (a, sa), (b, sb), (c2, sc) = exS[k], exW[k], exR[k]
    print(f"  {k:28s} SMT {a:.4f}+-{sa:.4f}  weighted {b:.4f}+-{sb:.4f} ({(a-b)/np.hypot(sa,sb):+.2f} SE)  resampled {c2:.4f}+-{sc:.4f}")
sw = dict(null=sliced_wasserstein(A.z, C.z), signal=sliced_wasserstein(C.z, C.z, wX=w),
          test=sliced_wasserstein(C.z, SMT.z, wX=w), resampled_vs_smt=sliced_wasserstein(zW, SMT.z))
print("sliced Wasserstein:", {k: round(v, 5) for k, v in sw.items()}, "  (test should sit at null)")
res["stages"]["arms"] = dict(
    calibrated=dict(m=int(csS.m), smt_vs_c_max=float(np.abs(devS).max()), smt_vs_c_within2=int((np.abs(devS)<=2).sum()),
                    w_vs_c_max=float(np.abs(devW).max()), smt_vs_w_max=float(np.abs(dSW).max()), smt_vs_w_within2=int((np.abs(dSW)<=2).sum())),
    heldout=dict(m=int(hoS.m), max=float(np.abs(dh).max()), within2=int((np.abs(dh)<=2).sum())),
    exotics={k: dict(smt=[float(exS[k][0]), float(exS[k][1])], weighted=[float(exW[k][0]), float(exW[k][1])],
                     resampled=[float(exR[k][0]), float(exR[k][1])]) for k in EXOTICS},
    sw=sw, rejections_smt=int(SMT.n_rejected), ess_C=float(eC["ess_frac"])); save()

### Stage 5 — resampling spread and the step-count series

In [ ]:
from taskc.smt import sample_strided
reps = 2 if QUICK else 5
pay = {k: [] for k in EXOTICS}
for i in range(reps):
    pi = ts.std.to_paths(resample_weighted(C.z, w, N_EVAL, seed=3000+i))
    for k in EXOTICS: pay[k].append(ev.price_exotics(pi)[k][0])
print("arm W resampling spread over", reps, "reps:", {k: round(float(np.std(v, ddof=1)), 5) for k, v in pay.items()})
res["stages"]["resample_spread"] = {k: dict(mean=float(np.mean(v)), sd=float(np.std(v, ddof=1))) for k, v in pay.items()}
sd = csC.G.std(0); series = {}
for K in ((sched.t_start+1, 100) if QUICK else (sched.t_start+1, 500, 250, 100, 50)):
    dS = sample_strided(model, sched, 5_000 if QUICK else 20_000, K, seed=4000+K, cfg=RUN, device=DEVICE, hnet=hnet)
    gS = build(ts.std.to_paths(dS.z), spec["testfuns"], spec["vanillas"], q).G
    series[K] = float(np.abs((gS.mean(0)-csS.c)/sd).max()); print(f"  K={K:4d}: SMT max |E[g]-c|/sd = {series[K]:.4f}")
res["stages"]["step_series"] = {str(k): v for k, v in series.items()}
res["timings"]["total_s"] = float(sum(v for v in res["timings"].values())); save()
print(f"\ntotal {res['timings']['total_s']/60:.1f} min -> {OUT}")

Results in `amortization.json` (every stage, with the fp32 flags, the EMA drift, the gate table and the compute budget). Numbers go to `DECISIONS.md` §13 before the manuscript is touched.